# LangGraph Basics: A Tiny Calculator Graph

This notebook introduces **LangGraph** with a simple calculator.

No LLM calls. No API key. No external services.

We will build a graph that can:

- add two numbers
- multiply two numbers
- route with conditional edges
- draw the graph
- inspect the graph execution

## Learning Objectives

By the end of this notebook, you should be able to:

1. Explain what LangGraph is.
2. Define graph state.
3. Create graph nodes.
4. Add normal edges.
5. Add conditional edges.
6. Compile and run a graph.
7. Draw a graph.
8. Understand how LangGraph relates to agent workflows.

## What Is LangGraph?

LangGraph is a framework for building graph-based workflows.

It is useful when you want more control than a simple chain:

```text
Simple chain:

Step 1 -> Step 2 -> Step 3
```

LangGraph supports branching and loops:

```text
Start
  |
  v
Decision
  |        \
  v         v
Path A    Path B
  \        /
   v      v
     End
```

In GenAI apps, LangGraph is often used for agents and multi-step workflows. Today we use it without an LLM so the basics are easier to see.

## Core Concepts

LangGraph has a few important ideas:

```text
State
  Shared data passed through the graph.

Node
  A Python function that reads state and returns updates.

Edge
  A connection from one node to another.

Conditional edge
  A decision that chooses the next node.

START and END
  Special markers for graph entry and exit.
```

## Setup

This notebook uses only LangGraph and standard Python.

If LangGraph is not installed, run the install cell below once, then restart the kernel.

In [ ]:
# Run only if LangGraph is not installed.
# After installing, restart the kernel and run the notebook from the top.

#%pip install -U "langgraph"

## Import and Check Versions

The next cell imports the LangGraph pieces we need.

In [ ]:
from importlib.metadata import version
from typing import TypedDict
from IPython.display import Image, Markdown, display

from langgraph.graph import END, START, StateGraph

print("langgraph:", version("langgraph"))
print("Imports are ready.")

langgraph: 1.2.11
Imports are ready.


## Our Calculator Graph

The calculator receives state like this:

```python
{
    "operation": "add",
    "a": 10,
    "b": 5
}
```

The graph decides which node should run:

```text
START
  |
  v
choose operation
  | add
  v
add node
  |
  v
format result
  |
  v
END
```

For multiplication, the graph takes a different path.

## Define the State

State is the shared data that moves through the graph.

We use `TypedDict` so the expected keys are easy to read.

`total=False` means not every key must exist at the beginning. For example, `result` does not exist until after a calculation node runs.

In [ ]:
class CalculatorState(TypedDict, total=False):
    operation: str
    a: float
    b: float
    result: float
    explanation: str
    display: str
    error: str

## Define Nodes

Each node is a normal Python function.

A node receives the current state and returns a dictionary with updates.

```text
current state -> node function -> state updates
```

In [ ]:
def add_numbers(state: CalculatorState) -> dict:
    result = state["a"] + state["b"]
    return {
        "result": result,
        "explanation": f'Added {state["a"]} + {state["b"]} = {result}',
    }


def multiply_numbers(state: CalculatorState) -> dict:
    result = state["a"] * state["b"]
    return {
        "result": result,
        "explanation": f'Multiplied {state["a"]} * {state["b"]} = {result}',
    }


def format_result(state: CalculatorState) -> dict:
    return {
        "display": f'Result: {state["result"]}. {state["explanation"]}.',
    }


def unsupported_operation(state: CalculatorState) -> dict:
    operation = state.get("operation", "missing")
    return {
        "error": f"Unsupported operation: {operation}",
        "display": "Please choose 'add' or 'multiply'.",
    }

## Define the Conditional Router

This function decides where the graph should go next.

It returns a label, and LangGraph maps that label to a node.

In [ ]:
def choose_operation(state: CalculatorState) -> str:
    operation = state.get("operation")

    if operation == "add":
        return "add"

    if operation == "multiply":
        return "multiply"

    return "unsupported"

## Build the Graph

Now we connect everything.

Graph shape:

```text
                +------+
START --add---->| add  |----+
   \            +------+    |
    \                       v
     \                  +--------+
      \--multiply------>| format |----> END
       \                +--------+
        \
         \unsupported
          v
       +-------------+
       | unsupported |----> END
       +-------------+
```

In [ ]:
graph_builder = StateGraph(CalculatorState)

graph_builder.add_node("add", add_numbers)
graph_builder.add_node("multiply", multiply_numbers)
graph_builder.add_node("format", format_result)
graph_builder.add_node("unsupported", unsupported_operation)

graph_builder.add_conditional_edges(
    START,
    choose_operation,
    {
        "add": "add",
        "multiply": "multiply",
        "unsupported": "unsupported",
    },
)

graph_builder.add_edge("add", "format")
graph_builder.add_edge("multiply", "format")
graph_builder.add_edge("format", END)
graph_builder.add_edge("unsupported", END)

calculator_graph = graph_builder.compile()

print("Graph compiled.")

Graph compiled.


## Draw the Graph

LangGraph can draw graphs in a few ways:

- `draw_mermaid()` returns Mermaid text.
- `draw_mermaid_png()` returns a PNG image that looks closer to a diagram.
- `draw_png()` can draw locally, but usually requires `pygraphviz`.

The next cell tries to show the PNG version first. If the environment cannot reach the Mermaid rendering service, it falls back to Mermaid text.

In [ ]:
graph_visual = calculator_graph.get_graph()
mermaid_graph = graph_visual.draw_mermaid()

try:
    png_bytes = graph_visual.draw_mermaid_png()
    display(Image(png_bytes))
except Exception as exc:
    print("PNG rendering is not available in this environment.")
    print("Showing Mermaid text instead.")
    print("Reason:", exc)

display(Markdown(f"```mermaid\n{mermaid_graph}\n```"))

PNG rendering is not available in this environment.
Showing Mermaid text instead.
Reason: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`


```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	add(add)
	multiply(multiply)
	format(format)
	unsupported(unsupported)
	__end__([<p>__end__</p>]):::last
	__start__ -.-> add;
	__start__ -.-> multiply;
	__start__ -.-> unsupported;
	add --> format;
	multiply --> format;
	format --> __end__;
	unsupported --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

## Run the Graph: Add

We call `.invoke(...)` with the initial state.

LangGraph returns the final state.

In [ ]:
add_input = {
    "operation": "add",
    "a": 12,
    "b": 8,
}

add_output = calculator_graph.invoke(add_input)
add_output

{'operation': 'add',
 'a': 12,
 'b': 8,
 'result': 20,
 'explanation': 'Added 12 + 8 = 20',
 'display': 'Result: 20. Added 12 + 8 = 20.'}

## Run the Graph: Multiply

The same graph takes a different path when the operation is `multiply`.

In [ ]:
multiply_input = {
    "operation": "multiply",
    "a": 12,
    "b": 8,
}

multiply_output = calculator_graph.invoke(multiply_input)
multiply_output

{'operation': 'multiply',
 'a': 12,
 'b': 8,
 'result': 96,
 'explanation': 'Multiplied 12 * 8 = 96',
 'display': 'Result: 96. Multiplied 12 * 8 = 96.'}

## Run the Graph: Unsupported Operation

This demonstrates the third conditional path.

In [ ]:
bad_input = {
    "operation": "divide",
    "a": 12,
    "b": 8,
}

bad_output = calculator_graph.invoke(bad_input)
bad_output

{'operation': 'divide',
 'a': 12,
 'b': 8,
 'display': "Please choose 'add' or 'multiply'.",
 'error': 'Unsupported operation: divide'}

## Stream the Graph Step by Step

`.stream(...)` lets us inspect what each node returns as the graph runs.

This is very useful for debugging.

In [ ]:
for step in calculator_graph.stream({"operation": "multiply", "a": 3, "b": 7}):
    print(step)

{'multiply': {'result': 21, 'explanation': 'Multiplied 3 * 7 = 21'}}
{'format': {'display': 'Result: 21. Multiplied 3 * 7 = 21.'}}


: 

## What Just Happened?

For `operation = "add"`:

```text
START -> add -> format -> END
```

For `operation = "multiply"`:

```text
START -> multiply -> format -> END
```

For another operation:

```text
START -> unsupported -> END
```

That routing is controlled by `add_conditional_edges(...)`.

## Why This Matters for GenAI

Today we used a calculator with no LLM.

In real GenAI apps, nodes might do things like:

- call an LLM
- retrieve documents
- call a tool
- validate output
- ask a human for approval
- decide whether to retry

LangGraph is useful because it makes the control flow explicit.

```text
State + Nodes + Edges + Conditions = controllable workflow
```

## Key Takeaways

- LangGraph builds graph-based workflows.
- State is shared across nodes.
- Nodes are Python functions.
- Edges connect nodes.
- Conditional edges choose a path at runtime.
- `.compile()` creates a runnable graph.
- `.invoke()` runs the graph and returns final state.
- `.stream()` shows intermediate node outputs.

## Exercises

### Exercise 1: Add Subtraction

Add a new node:

```python
def subtract_numbers(state):
    ...
```

Then update the router and conditional edge mapping.

### Exercise 2: Add Division

Add a division node.

If `b == 0`, route to an error node or return an error message.

### Exercise 3: Add a Validation Node

Add a node before the router that checks whether `a` and `b` are numbers.

Possible graph:

```text
START -> validate -> route -> add/multiply/error -> format -> END
```

### Exercise 4: Change the Output

Update `format_result` so it returns:

```text
The answer is 20.
```

### Exercise 5: Explain the Graph

In your own words, explain:

1. What is state?
2. What is a node?
3. What is a conditional edge?
4. Why is LangGraph useful for agent workflows?